# 0. Sobre — Gold UF

**Objetivo:** realizar um diagnóstico inicial de estrutura, qualidade, cobertura e distribuição dos dados, antes da integração das bases do Tech Challenge 3.

**Dicionário de dados:** preencher com as definições das variáveis relevantes após inspecionar a base.

**Perguntas norteadoras:** a estrutura está consistente? Há problemas de qualidade? A cobertura é adequada? Há valores atípicos que exigem investigação?

# 1. Configuração e importação de bibliotecas

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# 2. Carga e inspeção dos dados

Verificar dimensões, tipos de colunas e primeiros registros.

In [3]:
caminho = Path("../data/raw/gold/alfabetizacao_uf")

df = pd.read_parquet(caminho)

print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
display(df.head())
df.info()

Linhas: 145 | Colunas: 42


,serie,rede,sigla_uf,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,...,gap_meta_2030,atingiu_meta_2024,atingiu_meta_2025,atingiu_meta_2026,atingiu_meta_2027,atingiu_meta_2028,atingiu_meta_2029,atingiu_meta_2030,gold_processed_at,ano
0,2,5,MA,56.43,749.9153,NaN,NaN,NaN,NaN,NaN,...,-23.57,False,False,False,False,False,False,False,2026-10-07 15:38:11.194313,2023
1,2,3,GO,66.72,752.1315,NaN,NaN,NaN,NaN,NaN,...,-13.28,False,False,False,False,False,False,False,2026-10-07 15:38:11.194313,2023
2,2,5,MT,55.03,746.8000,NaN,NaN,NaN,NaN,NaN,...,-24.97,False,False,False,False,False,False,False,2026-10-07 15:38:11.194313,2023
3,2,5,MS,47.45,735.9520,NaN,NaN,NaN,NaN,NaN,...,-32.55,False,False,False,False,False,False,False,2026-10-07 15:38:11.194313,2023
4,2,3,TO,43.72,734.2853,NaN,NaN,NaN,NaN,NaN,...,-36.28,False,False,False,False,False,False,False,2026-10-07 15:38:11.194313,2023


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 42 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   serie                         145 non-null    int32         
 1   rede                          145 non-null    int32         
 2   sigla_uf                      145 non-null    object        
 3   taxa_alfabetizacao            145 non-null    float64       
 4   media_portugues               145 non-null    float64       
 5   proporcao_aluno_nivel_0       75 non-null     float64       
 6   proporcao_aluno_nivel_1       75 non-null     float64       
 7   proporcao_aluno_nivel_2       75 non-null     float64       
 8   proporcao_aluno_nivel_3       75 non-null     float64       
 9   proporcao_aluno_nivel_4       75 non-null     float64       
 10  proporcao_aluno_nivel_5       75 non-null     float64       
 11  proporcao_aluno_nivel_6       75

# 3. Qualidade dos dados

Consolidar valores ausentes e duplicatas em uma única verificação. Diferenciar duplicata exata de repetição de chave de negócio.

In [ ]:
qualidade = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True)
}).sort_values("nulos_%", ascending=False)

display(qualidade)
print("Linhas duplicadas exatas:", df.duplicated().sum())

## 3.1 Chaves e regras de negócio

Validar somente as chaves que representem efetivamente a granularidade da base. **A chave abaixo é uma hipótese de trabalho e deve ser confirmada com o dicionário de dados.**

In [ ]:
chave_proposta = ['ano', 'sigla_uf', 'rede']
if chave_proposta and all(c in df.columns for c in chave_proposta):
    print("Nulos na chave:", df[chave_proposta].isna().any(axis=1).sum())
    print("Repetições da chave proposta:", df.duplicated(chave_proposta).sum())
else:
    print("Chave não definida ou colunas ausentes: revisar granularidade antes do diagnóstico.")

In [ ]:
numericas = df.select_dtypes(include="number").columns
if len(numericas):
    display(df[numericas].describe().T.round(2))
else:
    print("Nenhuma coluna numérica identificada.")

### Observações de qualidade

- **Inconsistências identificadas:** preencher após executar.
- **Regras adicionais:** aplicar conforme definição oficial dos indicadores; não tratar automaticamente zeros ou negativos como erros.

# 4. Análise exploratória dos dados

Verificar cobertura e visualizar somente os indicadores úteis para esta base.

In [ ]:
dimensoes = ['ano', 'sigla_uf', 'rede']
for coluna in dimensoes:
    if coluna in df.columns:
        valores = df[coluna].dropna()
        print(f"{coluna}: {valores.nunique():,} valores distintos")
        if coluna == "ano":
            display(df[coluna].value_counts(dropna=False).sort_index().rename("registros"))
        else:
            display(df[coluna].value_counts(dropna=False).head(10).rename("registros"))

## 4.1 Distribuição de taxa de alfabetização

Selecionar um indicador representativo, evitando múltiplos gráficos com a mesma informação.

In [ ]:
indicador = 'taxa_alfabetizacao'  # substituir se a coluna tiver outro nome
if indicador and indicador in df.columns and pd.api.types.is_numeric_dtype(df[indicador]):
    display(df[indicador].describe().round(2))
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=indicador, bins=25)
    plt.title(f"Distribuição — {indicador}")
    plt.tight_layout()
    plt.show()
else:
    print("Defina um indicador numérico existente para visualizar sua distribuição.")

## 4.2 Validações específicas

Investigar, quando aplicável, percentuais fora da escala, códigos inválidos, cobertura por ano, denominadores ausentes, valores incompatíveis e outliers relevantes. **Evitar regras genéricas não justificadas.**

In [ ]:
# Exemplo de regra: habilite APENAS se o indicador for uma taxa na escala 0–100.
# coluna_taxa = "taxa_alfabetizacao"
# if coluna_taxa in df.columns:
#     inconsistentes = df[coluna_taxa].notna() & ~df[coluna_taxa].between(0, 100)
#     print("Taxas fora do intervalo [0, 100]:", inconsistentes.sum())

# 5. Conclusão geral

**Estrutura e cobertura:** preencher com os achados reais.

**Qualidade dos dados:** listar somente problemas efetivamente encontrados.

**Impacto para a integração:** registrar chaves, diferenças de anos, nível de agregação e ajustes necessários.

**Decisão:** pronta para integração / exige investigação / exige tratamento (definir após execução).